### 

### 

##### 

In [2]:
# ============================================================
# CELL 1 — DOMAIN-ADAPTIVE PRETRAINING (DAPT)
# ============================================================
# PURPOSE:
# Adapt SahajBERT to:
# - Banglish
# - YouTube comments
# - emoticons
# - informal Bengali
# - social-media syntax
#
# METHOD:
# Masked Language Modeling (MLM)
#
# OUTPUT:
# ./dapt_sahajbert/
#
# IMPORTANT:
# This is UNSUPERVISED.
# NO labels used here.
# ============================================================

import os
import gc
import random
import numpy as np
import pandas as pd
import torch

os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

SEED = 42

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

set_seed(SEED)

DEVICE = torch.device(
    'cuda:0' if torch.cuda.is_available() else 'cpu'
)

print("DEVICE:", DEVICE)

# ============================================================
# SAFETY FIX
# ============================================================

import transformers.modeling_utils as mu
mu.check_torch_load_is_safe = lambda: None

# ============================================================
# IMPORTS
# ============================================================

from datasets import Dataset

from transformers import (

    AutoTokenizer,

    AutoModelForMaskedLM,

    DataCollatorForLanguageModeling,

    Trainer,

    TrainingArguments,

    EarlyStoppingCallback
)

# ============================================================
# BASE MODEL
# ============================================================

BASE_MODEL = 'neuropark/sahajBERT'

DEVICE: cuda:0


In [3]:
# ============================================================
# CELL 2 — LOAD 191K FOR DAPT
# ============================================================
# IMPORTANT:
# DAPT should use LIGHT preprocessing only
# NOT full EPH
# ============================================================

import re

df = pd.read_csv(
    './BanglishDetect/data/raw/youtube_collected_raw.csv'
)

print(f"Raw comments: {len(df):,}")

# ============================================================
# LIGHT CLEANING ONLY
# ============================================================

def light_clean(text):

    text = str(text)

    # remove urls
    text = re.sub(r'http\\S+', ' ', text)

    # remove excessive whitespace
    text = re.sub(r'\\s+', ' ', text)

    # trim
    text = text.strip()

    return text

print("\nApplying LIGHT preprocessing...")

df['text'] = (
    df['text']
    .fillna('')
    .astype(str)
    .apply(light_clean)
)

# ============================================================
# REMOVE EMPTY
# ============================================================

df = df[
    df['text'].str.strip().str.len() > 2
].reset_index(drop=True)

print(f"\nAfter cleaning: {len(df):,}")

# ============================================================
# REMOVE DUPLICATES
# ============================================================

df = df.drop_duplicates(
    subset=['text']
).reset_index(drop=True)

print(f"After deduplication: {len(df):,}")

# ============================================================
# OPTIONAL LENGTH FILTER
# ============================================================

df = df[
    df['text'].str.len() <= 500
].reset_index(drop=True)

print(f"After length filter: {len(df):,}")

print("\nSample texts:\n")

for i in range(5):
    print(df['text'].iloc[i])
    print()

C:\Users\AFA001\AppData\Local\Temp\ipykernel_3388\4179788935.py:11: DtypeWarning: Columns (16,18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Raw comments: 191,000

Applying LIGHT preprocessing...

After cleaning: 191,000
After deduplication: 191,000
After length filter: 189,694

Sample texts:

দেখুন ১বছর প্রতিক্ষার পর আরমান আলিফের এই ঈদে (২০২২) একমাত্র নতুন গানের মিউজিক ভিডিও 
😥 এক ফোঁটা কান্না 😥👉 https://youtu.be/GvMgjjNDh2Q

মানুষ বদলে জায়.. সময় চলে জায়.. হারিয়ে জায় কতো আপন জন।
শুধু থেকে জায় কিছু স্মৃতি আজিবন।।
গানটা আজো২০২৪ সালে এসেও বুকের বিতটা মুচর দিয়ে উঠে।।😢❤

- কিছুদিন যোগাযোগ বন্ধ করে দাও,😕

- দেখবে কাছের মানুষ গুলো হারিয়ে গেছে!🖤

অনেক অপেক্ষা করছি গানটার জন্য😪

অনেক ভালোবাসা রইল ভারত থেকে❤️

সেই ১৫ বছর আগের স্মৃতি মনে পড়ে গেল যে যাকে চায় তার সাথে তার ভাগ্যের অনেক ফারাক থাকে



In [4]:
# ============================================================
# CELL 3 — CREATE DAPT DATASET
# ============================================================

# IMPORTANT:
# MLM does NOT need labels

dapt_df = pd.DataFrame({
    'text': df['text']
})

print(dapt_df.head())

# ============================================================
# HF DATASET
# ============================================================

dapt_ds = Dataset.from_pandas(dapt_df)

print(dapt_ds)

                                                text
0  দেখুন ১বছর প্রতিক্ষার পর আরমান আলিফের এই ঈদে (...
1  মানুষ বদলে জায়.. সময় চলে জায়.. হারিয়ে জায় কতো ...
2  - কিছুদিন যোগাযোগ বন্ধ করে দাও,😕\n\n- দেখবে কা...
3  অনেক অপেক্ষা করছি গানটার জন্য😪\n\nঅনেক ভালোবাস...
4  সেই ১৫ বছর আগের স্মৃতি মনে পড়ে গেল যে যাকে চা...
Dataset({
    features: ['text'],
    num_rows: 189694
})


In [5]:
# ============================================================
# CELL 4 — TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

print("Tokenizer loaded")

Tokenizer loaded


In [6]:
# ============================================================
# CELL 5 — TOKENIZATION
# ============================================================

MAX_LEN = 128

def tokenize(batch):

    return tokenizer(

        batch['text'],

        truncation=True,

        padding='max_length',

        max_length=MAX_LEN
    )

print("\nTokenizing dataset...")

dapt_ds = dapt_ds.map(
    tokenize,
    batched=True,
    batch_size=1000
)

print(dapt_ds)


Tokenizing dataset...


Map:   0%|          | 0/189694 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 189694
})


In [7]:
# ============================================================
# CELL 6 — TORCH FORMAT
# ============================================================

dapt_ds.set_format(

    type='torch',

    columns=[
        'input_ids',
        'attention_mask'
    ]
)

print("Torch formatting complete")

Torch formatting complete


In [8]:
# ============================================================
# CELL 7 — MLM DATA COLLATOR
# ============================================================
# MLM randomly masks tokens during training
# ============================================================

data_collator = DataCollatorForLanguageModeling(

    tokenizer=tokenizer,

    mlm=True,

    mlm_probability=0.15
)

print("MLM collator ready")

MLM collator ready


In [9]:
# ============================================================
# CELL 8 — LOAD MLM MODEL
# ============================================================

model = AutoModelForMaskedLM.from_pretrained(
    BASE_MODEL
)

model.to(DEVICE)

print("MLM model loaded")

Loading weights:   0%|          | 0/30 [00:00<?, ?it/s]

[transformers] AlbertForMaskedLM LOAD REPORT from: neuropark/sahajBERT
Key                              | Status     |  | 
---------------------------------+------------+--+-
albert.pooler.weight             | UNEXPECTED |  | 
albert.pooler.bias               | UNEXPECTED |  | 
sop_classifier.classifier.bias   | UNEXPECTED |  | 
sop_classifier.classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MLM model loaded


In [10]:
# ============================================================
# CELL 9 — DAPT TRAINING ARGUMENTS
# ============================================================

training_args = TrainingArguments(

    output_dir='./dapt_sahajbert',

    eval_strategy='no',

    save_strategy='epoch',

    logging_strategy='steps',

    logging_steps=500,

    learning_rate=5e-5,

    per_device_train_batch_size=16,

    gradient_accumulation_steps=2,

    num_train_epochs=3,

    weight_decay=0.01,

    warmup_ratio=0.10,

    fp16=True,

    save_total_limit=2,

    seed=SEED,

    report_to='none'
)

print("Training arguments ready")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training arguments ready


In [11]:
# ============================================================
# CELL 10 — CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=dapt_ds,

    data_collator=data_collator
)

print("Trainer ready")

Trainer ready


In [12]:
# ============================================================
# CELL 11 — START DAPT TRAINING
# ============================================================

gc.collect()

torch.cuda.empty_cache()

print("\nStarting DAPT training...")

trainer.train()

print("\n✅ DAPT training complete")


Starting DAPT training...


Step,Training Loss
500,7.331619
1000,6.686108
1500,6.782487
2000,6.754348
2500,6.710150
3000,6.565287
3500,6.474970
4000,6.449748
4500,6.390813
5000,6.254371


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ DAPT training complete


In [13]:
# ============================================================
# CELL 12 — SAVE DAPT MODEL
# ============================================================

SAVE_PATH = './dapt_sahajbert'

trainer.save_model(SAVE_PATH)

tokenizer.save_pretrained(SAVE_PATH)

print("\n✅ DAPT model saved")
print("Saved to:", SAVE_PATH)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ DAPT model saved
Saved to: ./dapt_sahajbert


In [14]:
# ============================================================
# CELL 13 — LOAD DAPT MODEL FOR CLASSIFICATION
# ============================================================
# IMPORTANT:
# This converts DAPT model into classifier
# ============================================================

from transformers import (
    AutoModelForSequenceClassification
)

dapt_classifier = AutoModelForSequenceClassification.from_pretrained(

    './dapt_sahajbert',

    num_labels=2
)

dapt_classifier.to(DEVICE)

print("DAPT classifier loaded")

Loading weights:   0%|          | 0/23 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: ./dapt_sahajbert
Key                          | Status     | 
-----------------------------+------------+-
predictions.dense.weight     | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
classifier.weight            | MISSING    | 
albert.pooler.bias           | MISSING    | 
albert.pooler.weight         | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DAPT classifier loaded


In [18]:
# ============================================================
# REBUILD CLASSIFICATION DATASETS
# ============================================================

from datasets import Dataset

from sklearn.metrics import (

    accuracy_score,

    f1_score,

    precision_score,

    recall_score,

    matthews_corrcoef
)

# ============================================================
# LOAD GOLD SPLITS
# ============================================================

train_df = pd.read_csv(
    './BanglishDetect/data/processed/gold_train.csv'
)

val_df = pd.read_csv(
    './BanglishDetect/data/processed/gold_val.csv'
)

test_df = pd.read_csv(
    './BanglishDetect/data/processed/gold_test.csv'
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

# ============================================================
# HF DATASETS
# ============================================================

train_ds = Dataset.from_pandas(train_df)

val_ds = Dataset.from_pandas(val_df)

test_ds = Dataset.from_pandas(test_df)

# ============================================================
# TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    './dapt_sahajbert'
)

# ============================================================
# TOKENIZATION
# ============================================================

def tokenize(batch):

    return tokenizer(

        batch['text'],

        truncation=True,

        padding='max_length',

        max_length=128
    )

train_ds = train_ds.map(
    tokenize,
    batched=True
)

val_ds = val_ds.map(
    tokenize,
    batched=True
)

test_ds = test_ds.map(
    tokenize,
    batched=True
)

cols = [
    'input_ids',
    'attention_mask',
    'label'
]

train_ds.set_format(
    type='torch',
    columns=cols
)

val_ds.set_format(
    type='torch',
    columns=cols
)

test_ds.set_format(
    type='torch',
    columns=cols
)

# ============================================================
# METRICS
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return {

        'accuracy':
            accuracy_score(labels, preds),

        'f1':
            f1_score(labels, preds),

        'precision':
            precision_score(labels, preds),

        'recall':
            recall_score(labels, preds),

        'mcc':
            matthews_corrcoef(labels, preds)
    }

print("\n✅ Classification datasets rebuilt")

(700, 2)
(150, 2)
(150, 2)


Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]


✅ Classification datasets rebuilt


In [19]:
# ============================================================
# CELL 14 — TRAIN DAPT MODEL ON GOLD DATA
# ============================================================

# Reuse:
# train_ds
# val_ds
# test_ds
# compute_metrics

classification_args = TrainingArguments(

    output_dir='./dapt_classifier_results',

    eval_strategy='epoch',

    save_strategy='epoch',

    logging_strategy='epoch',

    learning_rate=1e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=10,

    weight_decay=0.01,

    warmup_steps=100,

    load_best_model_at_end=True,

    metric_for_best_model='f1',

    greater_is_better=True,

    fp16=True,

    seed=SEED,

    report_to='none'
)

classifier_trainer = Trainer(

    model=dapt_classifier,

    args=classification_args,

    train_dataset=train_ds,

    eval_dataset=val_ds,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

gc.collect()

torch.cuda.empty_cache()

print("\nTraining DAPT classifier...")

classifier_trainer.train()

print("\nEvaluating DAPT model...")

dapt_results = classifier_trainer.evaluate(
    test_ds
)

print("\nDAPT RESULTS:")

for k, v in dapt_results.items():
    print(f"{k}: {v}")


Training DAPT classifier...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall,Mcc
1,0.643961,0.491008,0.820000,0.793893,0.764706,0.825397,0.636001
2,0.376125,0.363665,0.840000,0.809524,0.809524,0.809524,0.671593
3,0.272169,0.401967,0.846667,0.824427,0.794118,0.857143,0.690267
4,0.156579,0.451670,0.846667,0.816000,0.822581,0.809524,0.684650
5,0.083299,0.621412,0.840000,0.781818,0.914894,0.682540,0.677335


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating DAPT model...


Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall,Mcc
0.083299,0.453598,5,0.806667,0.760331,0.779661,0.741935,0.599003



DAPT RESULTS:
eval_loss: 0.4535983204841614
eval_accuracy: 0.8066666666666666
eval_f1: 0.7603305785123967
eval_precision: 0.7796610169491526
eval_recall: 0.7419354838709677
eval_mcc: 0.5990031796138748


In [20]:
# ============================================================
# CELL 15 — SAVE FINAL DAPT CLASSIFIER
# ============================================================

classifier_trainer.save_model(
    './final_dapt_classifier'
)

tokenizer.save_pretrained(
    './final_dapt_classifier'
)

print("\n✅ Final DAPT classifier saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Final DAPT classifier saved


In [ ]:
# ============================================================
# CELL — MULTI-SEED STATISTICAL EVALUATION
# ============================================================
# PURPOSE:
# Compare:
# 1. Original SahajBERT
# 2. DAPT-SahajBERT
#
# USING:
# - multiple seeds
# - mean ± std
# - paired t-test
# - Cohen's d
#
# OUTPUT:
# publication-ready statistical evidence
# ============================================================

import os
import gc
import torch
import random
import numpy as np
import pandas as pd

from scipy.stats import ttest_rel

from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    matthews_corrcoef
)

import transformers.modeling_utils as mu
mu.check_torch_load_is_safe = lambda: None

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = torch.device(
    'cuda:0' if torch.cuda.is_available() else 'cpu'
)

print("DEVICE:", DEVICE)

# ============================================================
# SEEDS
# ============================================================

SEEDS = [42, 123, 999]

# ============================================================
# MODELS
# ============================================================

MODELS = {

    'baseline':
        './best_sahajbert_gold',

    'dapt':
        './dapt_sahajbert'
}

# ============================================================
# LOAD GOLD SPLITS
# ============================================================

train_df = pd.read_csv(
    './BanglishDetect/data/processed/gold_train.csv'
)

val_df = pd.read_csv(
    './BanglishDetect/data/processed/gold_val.csv'
)

test_df = pd.read_csv(
    './BanglishDetect/data/processed/gold_test.csv'
)

# ============================================================
# TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    './dapt_sahajbert'
)

# ============================================================
# HF DATASETS
# ============================================================

train_ds = Dataset.from_pandas(train_df)

val_ds = Dataset.from_pandas(val_df)

test_ds = Dataset.from_pandas(test_df)

# ============================================================
# TOKENIZATION
# ============================================================

def tokenize(batch):

    return tokenizer(

        batch['text'],

        truncation=True,

        padding='max_length',

        max_length=128
    )

train_ds = train_ds.map(
    tokenize,
    batched=True
)

val_ds = val_ds.map(
    tokenize,
    batched=True
)

test_ds = test_ds.map(
    tokenize,
    batched=True
)

cols = [
    'input_ids',
    'attention_mask',
    'label'
]

train_ds.set_format(
    type='torch',
    columns=cols
)

val_ds.set_format(
    type='torch',
    columns=cols
)

test_ds.set_format(
    type='torch',
    columns=cols
)

# ============================================================
# METRICS
# ============================================================

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    return {

        'accuracy':
            accuracy_score(labels, preds),

        'f1':
            f1_score(labels, preds),

        'precision':
            precision_score(labels, preds),

        'recall':
            recall_score(labels, preds),

        'mcc':
            matthews_corrcoef(labels, preds)
    }

# ============================================================
# SEED FUNCTION
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ============================================================
# RESULTS STORAGE
# ============================================================

all_results = []

# ============================================================
# MAIN LOOP
# ============================================================

for model_name, model_path in MODELS.items():

    print("\n================================================")
    print("MODEL:", model_name)
    print("================================================")

    for seed in SEEDS:

        print(f"\nRunning seed: {seed}")

        set_seed(seed)

        gc.collect()

        torch.cuda.empty_cache()

        # ====================================================
        # LOAD MODEL
        # ====================================================

        model = AutoModelForSequenceClassification.from_pretrained(

            model_path,

            num_labels=2
        )

        model.to(DEVICE)

        # ====================================================
        # TRAINING ARGS
        # ====================================================

        training_args = TrainingArguments(

    output_dir=f'./tmp_{MODEL_TYPE}_{SEED}',

    eval_strategy='epoch',

    save_strategy='no',

    logging_strategy='epoch',

    learning_rate=1e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    gradient_accumulation_steps=4,

    num_train_epochs=5,

    warmup_steps=100,

    weight_decay=0.01,

    fp16=False,

    dataloader_num_workers=0,

    seed=SEED,

    report_to='none'
)

        # ====================================================
        # TRAINER
        # ====================================================

        trainer = Trainer(

            model=model,

            args=training_args,

            train_dataset=train_ds,

            eval_dataset=val_ds,

            compute_metrics=compute_metrics,

            callbacks=[
                EarlyStoppingCallback(
                    early_stopping_patience=2
                )
            ]
        )

        # ====================================================
        # TRAIN
        # ====================================================

        trainer.train()

        # ====================================================
        # EVALUATE
        # ====================================================

        results = trainer.evaluate(test_ds)

        row = {

            'model': model_name,

            'seed': seed,

            'accuracy':
                results['eval_accuracy'],

            'f1':
                results['eval_f1'],

            'precision':
                results['eval_precision'],

            'recall':
                results['eval_recall'],

            'mcc':
                results['eval_mcc']
        }

        all_results.append(row)

        print("\nRESULT:")
        print(row)

# ============================================================
# RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(all_results)

print("\n================================================")
print("ALL RESULTS")
print("================================================")

print(results_df)

# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(
    './multi_seed_results.csv',
    index=False
)

print("\n✅ Results saved")